# 01 — Coleta e Limpeza de Dados

**Objetivo:** Baixar os dados brutos de acidentes da PRF, filtrar para Niterói/RJ e gerar um dataset limpo para as análises seguintes.

**Fonte:** [PRF — Dados Abertos](https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf)

---

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Cria pastas se não existirem
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

print('Pastas criadas com sucesso!')

Pastas criadas com sucesso!


## 1. Carregamento dos Dados Brutos

> **Como baixar:** Acesse https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf  
> Baixe os arquivos CSV de acidentes agrupados por ocorrência (2021, 2022, 2023, 2024).  
> Salve em `../data/raw/` com os nomes: `acidentes_2021.csv`, `acidentes_2022.csv`, etc.

In [2]:
anos = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
dfs = []

for ano in anos:
    caminho = f'../data/raw/acidentes_{ano}.csv'
    if os.path.exists(caminho):
        # A PRF usa encoding latin-1 e separador ';'
        df_ano = pd.read_csv(caminho, encoding='latin-1', sep=';', low_memory=False)
        df_ano['ano_arquivo'] = ano
        dfs.append(df_ano)
        print(f'[{ano}] {len(df_ano):,} registros carregados')
    else:
        print(f'[{ano}] Arquivo não encontrado: {caminho}')

if dfs:
    df_raw = pd.concat(dfs, ignore_index=True)
    print(f'\nTotal combinado: {len(df_raw):,} registros')
    print(f'Colunas: {df_raw.columns.tolist()}')
else:
    print('Nenhum arquivo encontrado. Baixe os dados conforme as instruções acima.')

[2020] 63,585 registros carregados
[2021] 64,567 registros carregados
[2022] 64,606 registros carregados
[2023] 67,766 registros carregados
[2024] 73,156 registros carregados
[2025] 72,529 registros carregados
[2026] 11,380 registros carregados

Total combinado: 417,589 registros
Colunas: ['id', 'data_inversa', 'dia_semana', 'horario', 'uf', 'br', 'km', 'municipio', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'ignorados', 'feridos', 'veiculos', 'latitude', 'longitude', 'regional', 'delegacia', 'uop', 'ano_arquivo']


In [3]:
# Visão geral do dataset bruto
df_raw.head(3)

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop,ano_arquivo
0,260068.0,2020-01-01,quarta-feira,05:40:00,PA,316,84,SAO FRANCISCO DO PARA,Falta de Atenção à Condução,Saída de leito carroçável,...,0,1,2,2,"-1,3101929","-47,74456398",SPRF-PA,DEL01-PA,UOP02-DEL01-PA,2020
1,260073.0,2020-01-01,quarta-feira,06:00:00,MG,262,804,UBERABA,Falta de Atenção à Condução,Colisão transversal,...,3,0,1,2,"-19,76747537","-47,98725511",SPRF-MG,DEL13-MG,UOP01-DEL13-MG,2020
2,260087.0,2020-01-01,quarta-feira,06:00:00,BA,116,191,CANUDOS,Condutor Dormindo,Saída de leito carroçável,...,0,2,0,3,"-10,32002103","-39,06425211",SPRF-BA,DEL07-BA,UOP02-DEL07-BA,2020


In [4]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 417589 entries, 0 to 417588
Data columns (total 31 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      417589 non-null  float64
 1   data_inversa            417589 non-null  object 
 2   dia_semana              417589 non-null  object 
 3   horario                 417589 non-null  object 
 4   uf                      417589 non-null  object 
 5   br                      417589 non-null  int64  
 6   km                      417589 non-null  object 
 7   municipio               417589 non-null  object 
 8   causa_acidente          417589 non-null  object 
 9   tipo_acidente           417588 non-null  object 
 10  classificacao_acidente  417581 non-null  object 
 11  fase_dia                417589 non-null  object 
 12  sentido_via             417589 non-null  object 
 13  condicao_metereologica  417589 non-null  object 
 14  tipo_pista          

## 2. Filtro Geográfico — Niterói e Região

Filtramos os registros do estado do RJ, focando em Niterói e municípios limítrofes relevantes.

In [8]:
# Verifica quais colunas existem para filtrar por município
colunas_municipio = [c for c in df_raw.columns if 'munic' in c.lower() or 'cidade' in c.lower()]
print('Colunas de município encontradas:', colunas_municipio)

colunas_uf = [c for c in df_raw.columns if 'uf' in c.lower() or 'estado' in c.lower()]
print('Colunas de UF encontradas:', colunas_uf)

Colunas de município encontradas: ['municipio']
Colunas de UF encontradas: ['uf']


In [9]:
# Adapte os nomes das colunas conforme o resultado acima
COL_UF = 'uf'           # ajuste se necessário
COL_MUN = 'municipio'   # ajuste se necessário

municipios_alvo = [
    'NITEROI', 'SAO GONCALO', 'ITABORAI',
    'MARICA', 'RIO BONITO'
]

df_rj = df_raw[df_raw[COL_UF].str.upper() == 'RJ'].copy()
print(f'RJ total: {len(df_rj):,} registros')

df_niteroi = df_rj[
    df_rj[COL_MUN].str.upper().str.strip().isin(municipios_alvo)
].copy()

print(f'Região de Niterói: {len(df_niteroi):,} registros')
print(df_niteroi[COL_MUN].value_counts())

RJ total: 32,914 registros
Região de Niterói: 4,324 registros
municipio
SAO GONCALO    1559
ITABORAI       1223
NITEROI        1057
RIO BONITO      485
Name: count, dtype: int64


In [10]:
# Ver como a PRF grafa os municípios do RJ
municipios_rj = df_rj['municipio'].unique()
municipios_rj.sort()

# Filtra só os que nos interessam
import re
for m in sorted(municipios_rj):
    if any(x in m.upper() for x in ['NITER', 'GONCALO', 'ITABO', 'MARIC', 'BONITO']):
        print(repr(m))

'ITABORAI'
'NITEROI'
'RIO BONITO'
'SAO GONCALO'


## 3. Limpeza dos Dados

In [13]:
# --- 3.1 Valores nulos ---
nulos = df_niteroi.isnull().sum()
nulos_pct = (nulos / len(df_niteroi) * 100).round(2)

resumo_nulos = pd.DataFrame({'nulos': nulos, 'pct': nulos_pct})
print(resumo_nulos[resumo_nulos['nulos'] > 0].sort_values('pct', ascending=False))

           nulos   pct
delegacia      3  0.07
uop            3  0.07
regional       1  0.02


In [14]:
df = df_niteroi.copy()

# --- 3.2 Datas e horários ---
# Adapte os nomes das colunas conforme seu dataset
if 'data_inversa' in df.columns:
    df['data'] = pd.to_datetime(df['data_inversa'], errors='coerce', dayfirst=True)
elif 'data' in df.columns:
    df['data'] = pd.to_datetime(df['data'], errors='coerce', dayfirst=True)

if 'horario' in df.columns:
    df['hora'] = pd.to_datetime(df['horario'], format='%H:%M:%S', errors='coerce').dt.hour

# Extrai variáveis temporais
df['ano']           = df['data'].dt.year
df['mes']           = df['data'].dt.month
df['dia_semana']    = df['data'].dt.dayofweek   # 0=segunda, 6=domingo
dias_pt = {0:'Segunda',1:'Terça',2:'Quarta',3:'Quinta',4:'Sexta',5:'Sábado',6:'Domingo'}
df['dia_semana_nome'] = df['dia_semana'].map(dias_pt)

print('Datas processadas:', df['data'].notna().sum(), 'válidas')

Datas processadas: 1732 válidas


In [15]:
# --- 3.3 Coordenadas geográficas ---
for col in ['latitude', 'longitude']:
    if col in df.columns:
        # A PRF usa vírgula como separador decimal
        df[col] = df[col].astype(str).str.replace(',', '.').str.strip()
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove linhas sem coordenadas
antes = len(df)
df = df.dropna(subset=['latitude', 'longitude'])
depois = len(df)
print(f'Removidos {antes - depois} registros sem coordenadas. Restam: {depois:,}')

# Sanity check: coordenadas dentro do RJ
df = df[
    (df['latitude'].between(-23.5, -22.5)) &
    (df['longitude'].between(-43.5, -42.5))
]
print(f'Após filtro de coordenadas válidas: {len(df):,} registros')

Removidos 0 registros sem coordenadas. Restam: 4,324
Após filtro de coordenadas válidas: 4,320 registros


In [16]:
# --- 3.4 Padronização de colunas categóricas ---
cols_cat = ['causa_acidente', 'tipo_acidente', 'classificacao_acidente',
            'condicao_metereologica', 'tipo_pista', 'fase_dia']

for col in cols_cat:
    if col in df.columns:
        df[col] = df[col].str.strip().str.upper()
        df[col] = df[col].replace({'(NULL)': np.nan, 'NA': np.nan, '': np.nan})

print('Categorias na coluna classificacao_acidente:')
if 'classificacao_acidente' in df.columns:
    print(df['classificacao_acidente'].value_counts())

Categorias na coluna classificacao_acidente:
classificacao_acidente
COM VÍTIMAS FERIDAS    3481
SEM VÍTIMAS             652
COM VÍTIMAS FATAIS      187
Name: count, dtype: int64


In [17]:
# --- 3.5 Criar coluna de gravidade (target do modelo ML) ---
# A classificação da PRF costuma ter: Sem Vítimas, Com Vítimas Feridas, Com Vítimas Fatais
mapa_gravidade = {
    'SEM VÍTIMAS': 0,
    'COM VÍTIMAS FERIDAS': 1,
    'COM VÍTIMAS FATAIS': 2,
    'IGNORADO': np.nan
}

if 'classificacao_acidente' in df.columns:
    df['gravidade'] = df['classificacao_acidente'].map(mapa_gravidade)
    print('Distribuição de gravidade:')
    print(df['gravidade'].value_counts(dropna=False))

Distribuição de gravidade:
gravidade
1.0    3481
0.0     652
2.0     187
Name: count, dtype: int64


## 4. Salva Dataset Limpo

In [18]:
caminho_saida = '../data/processed/acidentes_niteroi_limpo.csv'
df.to_csv(caminho_saida, index=False, encoding='utf-8')

print(f'Dataset salvo em: {caminho_saida}')
print(f'Shape final: {df.shape}')
print(f'Período: {df["data"].min().date()} a {df["data"].max().date()}')
df.head(3)

Dataset salvo em: ../data/processed/acidentes_niteroi_limpo.csv
Shape final: (4320, 37)
Período: 2020-01-01 a 2026-12-02


,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,...,regional,delegacia,uop,ano_arquivo,data,hora,ano,mes,dia_semana_nome,gravidade
137,261110.0,2020-01-06,0.0,01:05:00,RJ,101,322,NITEROI,FALTA DE ATENÇÃO À CONDUÇÃO,COLISÃO COM OBJETO ESTÁTICO,...,SPRF-RJ,DEL02-RJ,UOP01-DEL02-RJ,2020,2020-06-01,1,2020.0,6.0,Segunda,0.0
273,261809.0,2020-01-09,1.0,23:30:00,RJ,101,"309,4",SAO GONCALO,AGRESSÃO EXTERNA,COLISÃO COM OBJETO ESTÁTICO,...,SPRF-RJ,DEL02-RJ,UOP02-DEL02-RJ,2020,2020-09-01,23,2020.0,9.0,Terça,1.0
653,263611.0,2020-01-18,NaN,14:10:00,RJ,493,"0,7",SAO GONCALO,FALTA DE ATENÇÃO À CONDUÇÃO,COLISÃO TRASEIRA,...,SPRF-RJ,DEL02-RJ,UOP02-DEL02-RJ,2020,NaT,14,NaN,NaN,NaN,1.0
